# Create Master table

Create a master table by unioning healthy and disease tables in bronze. Adds a column for variance across all datasets.

Also does some minimal EDA, 

## Create Master Table

In [0]:
%sql


SELECT probe_id, sample_id, access, beta, probe_var as probe_var_dataset, "healthy" AS sample_group  FROM bronze.methylation.gse213478_beta_long LIMIT 10

In [0]:
%sql

SELECT probe_id, sample_id, access, beta, probe_var as probe_var_dataset, "disease" AS sample_group  FROM bronze.methylation.gse289137_beta_long LIMIT 10

In [0]:
%sql

CREATE OR REPLACE TABLE silver.methylation.methylation_beta
USING DELTA
PARTITIONED BY (probe_id)
AS
SELECT 
  probe_id,
  sample_id,
  access,
  beta AS beta,
  probe_var AS probe_var_dataset,
  "healthy" AS sample_group
FROM bronze.methylation.GSE213478_beta_long

UNION ALL

SELECT 
  probe_id,
  sample_id,
  access,
  beta AS beta,
  probe_var AS probe_var_dataset,
  "disease" AS sample_group
FROM bronze.methylation.GSE289137_beta_long;



In [0]:
%sql

SELECT * FROM silver.methylation.methylation_beta

## Compute stats

In [0]:
%sql
CREATE OR REPLACE TABLE silver.methylation.methylation_beta_stats
USING DELTA
AS
SELECT
  probe_id,
  COUNT(DISTINCT sample_group) AS dataset_count,
  COUNT(DISTINCT sample_id) AS sample_count,
  VARIANCE(beta) AS beta_variance
FROM silver.methylation.methylation_beta
GROUP BY probe_id;


In [0]:
%sql

SELECT * FROM silver.methylation.methylation_beta_stats

In [0]:
%sql
SELECT dataset_count, count(1) as tot_probes FROM silver.methylation.methylation_beta_stats GROUP BY dataset_count

In [0]:
%sql

SELECT dataset_count, count(1) as tot_probes FROM silver.methylation.methylation_beta_stats WHERE beta_variance > 0.05 GROUP BY dataset_count

## PCA of healthy/disease

We filter probes that have variance >0.05 and are present in both datasets. This corresponds to about 

In [0]:
from pyspark.sql.functions import col

# Step 1: Filter probe stats
stats_df = spark.table("silver.methylation.methylation_beta_stats")

probe_ids_df = stats_df.filter(
    (col("dataset_count") == 2) & (col("beta_variance") > 0.05)
).select("probe_id")

# Step 2: Filter the master table using a semi join
beta_df = spark.table("silver.methylation.methylation_beta")

filtered_df = beta_df.join(probe_ids_df, on="probe_id", how="semi")


In [0]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: Select and convert to pandas (safe after filtering)
df = filtered_df.select("sample_id", "probe_id", "beta", "sample_group").toPandas()

# Step 2: Pivot to matrix (samples as rows, probes as columns)
df_pivot = df.pivot(index="sample_id", columns="probe_id", values="beta").fillna(0)

# Step 3: Get sample group labels
sample_groups = df.drop_duplicates(subset="sample_id")[["sample_id", "sample_group"]].set_index("sample_id")
group_labels = sample_groups.loc[df_pivot.index, "sample_group"]

# Step 4: Run PCA
X = StandardScaler().fit_transform(df_pivot)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Step 5: Plot PCA
pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"], index=df_pivot.index)
pca_df["sample_group"] = group_labels.values

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="sample_group", palette="Set2", s=50)
plt.title("PCA of Methylation Beta Values")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.grid(True)
plt.legend(title="Sample Group")
plt.tight_layout()
plt.show()


In [0]:
df_pivot

In [0]:
df_pivot.describe()


In [0]:
output_path = "/Volumes/gold/methylation/methylation_features/methylation_beta_001.parquet"

df_pivot.to_parquet(output_path, index=False)